### Importing Libraries and Loading Environment Variables
In this section, we import necessary libraries and load environment variables for Binance API access.

In [ ]:
import pandas as pd
from binance.client import Client
from binance.exceptions import BinanceAPIException
import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("BINANCE_API_KEY")
api_secret = os.getenv("BINANCE_SECRET_KEY")

### Initializing Binance Client
Here, we create a Binance client instance using the API key and secret.

In [ ]:
client = Client(api_key, api_secret)
client

### Fetching Account Information
This code block retrieves account information, including balances, from Binance.

In [ ]:
account = client.get_account()
account

### Converting Account Balances to DataFrame
We convert the account balances into a Pandas DataFrame for easier manipulation and analysis.

In [ ]:
df = pd.DataFrame(account["balances"])
df

### Converting Balance Values to Numeric
This step ensures that the 'free' balance values are numeric, which is necessary for calculations.

In [ ]:
df.free = pd.to_numeric(df.free, errors="coerce")

### Filtering Assets with Non-Zero Balance
We filter the DataFrame to include only assets with a non-zero 'free' balance.

In [ ]:
current_assets = df.loc[df.free > 0]
current_assets

### Calculating Total Balances
This block calculates the total balance (free + locked) for each asset and stores it in a dictionary.

In [ ]:
# Fetch balances with more than zero balance
balances = {asset['asset']: float(asset['free']) + float(asset['locked'])
            for asset in client.get_account()['balances'] if float(asset['free']) + float(asset['locked']) > 0}

balances

### Fetching Current Market Prices
Here, we retrieve the current market prices for all trading pairs from Binance.

In [ ]:
# Get current prices for all symbols
prices = {price['symbol']: float(price['price']) for price in client.get_all_tickers()}
prices

### Estimating Asset Values in USDT
This code estimates the value of each asset in USDT using current market prices.

In [ ]:

# Estimate the USDT value of each asset
estimated_values = {}
for asset, balance in balances.items():
    if asset == 'USDT':
        estimated_values[asset] = balance
    elif f"{asset}USDT" in prices:
        estimated_values[asset] = balance * prices[f"{asset}USDT"]
    elif f"USDT{asset}" in prices:  # Handle reverse pairs
        estimated_values[asset] = balance / prices[f"USDT{asset}"]

estimated_values

### Identifying the Most Valuable Asset
Finally, we determine which asset has the highest estimated value in USDT.

In [ ]:

# Determine the asset with the highest estimated USDT value
most_valuable_symbol = max(estimated_values, key=estimated_values.get) if estimated_values else "No Transaction"
most_valuable_symbol